# Pydentic model to structure output

TODO: Create an agent that simulate en IT employer

In [23]:
from pydantic_ai import Agent
from dotenv import load_dotenv

load_dotenv()

employee_simulator_agent = Agent (
    "openrouter:openai/gpt-oss-20b:free",
    system_prompt="""You are an HR expert within IT field in Sweden within data science, data engineering,
machine learning, AI engineering. You will simulate IT employees.

Fields to include in output:
- name
- age
- gender
- job_title
- salary in SEK per month"""
)
# prompt = ""

result= await employee_simulator_agent.run("Simulate two employees")

result

AgentRunResult(output='**Employee 1**  \n- **Name:** Sofia Larsson  \n- **Age:** 34  \n- **Gender:** Female  \n- **Job Title:** Senior Data Engineer  \n- **Salary in SEK per month:** 95\u202f000  \n\n**Employee 2**  \n- **Name:** Erik Andersson  \n- **Age:** 28  \n- **Gender:** Male  \n- **Job Title:** Machine Learning Engineer  \n- **Salary in SEK per month:** 88\u202f000')

In [24]:
print(result.output)

**Employee 1**  
- **Name:** Sofia Larsson  
- **Age:** 34  
- **Gender:** Female  
- **Job Title:** Senior Data Engineer  
- **Salary in SEK per month:** 95 000  

**Employee 2**  
- **Name:** Erik Andersson  
- **Age:** 28  
- **Gender:** Male  
- **Job Title:** Machine Learning Engineer  
- **Salary in SEK per month:** 88 000


In [ ]:
#with open("simulated_employees.md", "w") as file:
 #   file.write(result.output)

UnicodeEncodeError: 'charmap' codec can't encode character '\u202f' in position 162: character maps to <undefined>

## Get more structured output
issue with above:
- output structure vary
- hard to work with the data e.g. compute mean of salaries

want:
- repatable structure

In [28]:

from pydantic import BaseModel, Field
from typing import Literal
from pydantic_ai import Agent


class EmployeeModel(BaseModel):
    name: str = Field(
        description="Mostly swedish names, but could be foreign names as well"
    )
    age: int = Field(description="age should be between 18 and 67")
    gender: Literal["Male", "Female"]
    experience_level: Literal["Entry", "Mid level", "Senior", "Expert"]
    job_title: str
    salary: int = Field(
        gte=30_000,
        lte=50_000,
        description="salary should be between 30k and 50k, the higher experience level, the higher salary",
    )


employee_simulator_agent = Agent(
    "openrouter:openai/gpt-oss-20b:free",
    system_prompt="""
You are an HR expert within IT field in Sweden within data science, data engineering,
machine learning, AI engineering. You will simulate IT employees.
""",
)

result = await employee_simulator_agent.run("Give me 3 employees", output_type=EmployeeModel)
result

C:\Users\aless\AppData\Local\Temp\ipykernel_18084\1413229117.py:14: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'gte', 'lte'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  salary: int = Field(


AgentRunResult(output=EmployeeModel(name='Oskar Nilsson', age=30, gender='Male', experience_level='Mid level', job_title='Data Engineer', salary=42000))

In [29]:
result.output


EmployeeModel(name='Oskar Nilsson', age=30, gender='Male', experience_level='Mid level', job_title='Data Engineer', salary=42000)

In [ ]:
result.output.salary+5000

47000

In [31]:
employee_simulator_agent = Agent(
    "openrouter:nvidia/nemotron-nano-12b-v2-vl:free",
    system_prompt="""
You are an HR expert within IT field in Sweden within data science, data engineering,
machine learning, AI engineering. You will simulate IT employees.
""", retries=1
)

result = await employee_simulator_agent.run("Give me 3 employees", output_type=list[EmployeeModel], )
result

AgentRunResult(output=[EmployeeModel(name='Erik Karlsson', age=35, gender='Male', experience_level='Senior', job_title='Data Scientist', salary=45000), EmployeeModel(name='Sofia Lindström', age=29, gender='Female', experience_level='Mid level', job_title='Machine Learning Engineer', salary=38000), EmployeeModel(name='Alexander Johansson', age=42, gender='Male', experience_level='Expert', job_title='AI Research Lead', salary=50000)])

In [36]:
# BaseModel -> dictionary
result.output[0].model_dump()

{'name': 'Erik Karlsson',
 'age': 35,
 'gender': 'Male',
 'experience_level': 'Senior',
 'job_title': 'Data Scientist',
 'salary': 45000}

TODO: 
- result.output make into list of dictionary
- create pandas dataframe based on this list
- export in cv file

In [48]:
employee_list = []

for employee in result.output:

    employee_list.append(employee.model_dump())
employee_list


[{'name': 'Erik Karlsson',
  'age': 35,
  'gender': 'Male',
  'experience_level': 'Senior',
  'job_title': 'Data Scientist',
  'salary': 45000},
 {'name': 'Sofia Lindström',
  'age': 29,
  'gender': 'Female',
  'experience_level': 'Mid level',
  'job_title': 'Machine Learning Engineer',
  'salary': 38000},
 {'name': 'Alexander Johansson',
  'age': 42,
  'gender': 'Male',
  'experience_level': 'Expert',
  'job_title': 'AI Research Lead',
  'salary': 50000}]

In [49]:
import pandas as pd 
df = pd.DataFrame(employee_list)
df

,name,age,gender,experience_level,job_title,salary
0,Erik Karlsson,35,Male,Senior,Data Scientist,45000
1,Sofia Lindström,29,Female,Mid level,Machine Learning Engineer,38000
2,Alexander Johansson,42,Male,Expert,AI Research Lead,50000


We want the avarage of the salary:


In [50]:
df["salary"].mean()

np.float64(44333.333333333336)

In [51]:
df.to_csv("simulate_empluyees.csv")